# NGSO SLS — Slice E: Live Spacetime Pull

**What this does:** Pulls the live NMTS network model + installed intents from a Spacetime/Minkowski instance, builds the constellation element array, runs the Slice-A H3 coverage engine on it, and compares predicted access against installed PathIntent routes.

**Read-only, Increment-1 scope:** no Create/Update/Delete. Keplerian motion fully supported; TLE/ephemeris flagged and skipped (Sgp4 deferred). NMTS→coverage path gated behind the **Increment-0 capability probe**, and a **first-contact smoke probe** stages the live RPCs one at a time.

**Run order:** Install → Capability probe → Connection form → **Smoke probe** → Build store → Pull+coverage → Viz → Compare → Record.

---

## Setup — Colab Secrets

Open the **🔑 Secrets** panel (left sidebar), add the entries below, and enable **Notebook access** for each. The form cell prefills from these — nothing is hardcoded.

| Secret name | Value |
|---|---|
| `SPACETIME_KEY_ID` | your API key id |
| `SPACETIME_USER_ID` | service-account user id / email |
| `KEY_DATA_B64` | **base64 of your private key file** (single line — see below) |
| `SPACETIME_URL` | *(optional)* overrides the URL field default |
| `SPACETIME_MODEL_URL` | *(optional)* only if the Model service is on a different host |
| `GITHUB_TOKEN` | *(optional)* only if cloning a private repo |

Produce the `KEY_DATA_B64` value locally (one line, no newlines) and paste it as the secret:

    base64 < private_key.key | tr -d '\n'
    # (GNU coreutils shortcut: base64 -w0 private_key.key)

> **SECURITY — never paste the raw key or any secret as a notebook literal or into a code cell.** The key is read from the `KEY_DATA_B64` secret, base64-decoded **in memory**, and (only for the modern auth path, which needs a file) written to a **RAM-backed `0600` temp file that is shredded when the store closes** — it never touches persistent disk and never appears in outputs or git.

In [ ]:
# === Install: spacetime-api (private index) + ngso_sls ===
# Re-running refreshes the ngso_sls clone to the latest commit. IMPORTANT: to LOAD updated package
# code into a warm kernel, do Runtime -> Restart session, then Run all (Python caches imports).

# 1. Live Spacetime API from the private Artifact Registry index
!pip install --upgrade spacetime-api --extra-index-url https://us-central1-python.pkg.dev/a5a-spacetime-artifacts/py-packages/simple --prefer-binary -q

# 2. ngso_sls — clone/pull + non-editable install (always refreshes to latest)
REPO_URL = "https://github.com/luca-aalyria/spacetime-sls.git"
import importlib, importlib.util, subprocess, sys, os, re

def _run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("$", cmd)
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
        raise RuntimeError(f"command failed (exit {p.returncode}) — see output above")

url = REPO_URL
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
    if _tok and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://{_tok}@")
except Exception:
    pass
repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
if not os.path.isdir(repo_dir):
    _run(f"git clone {url} {repo_dir}")
else:
    _run(f"git -C {repo_dir} pull --ff-only")      # always refresh to the latest commit
_run(f"{sys.executable} -m pip install {os.path.abspath(repo_dir)} -q")
sys.path.insert(0, os.path.abspath(repo_dir))
importlib.invalidate_caches()

# 2b. vendored NBI (NetOps) stubs — bazel-layout fallback until the official pip package
# ships nbi_pb2 again (see vendor/spacetime_api_stubs/README.md). The pip root wins if present.
_stubs = os.path.join(os.path.abspath(repo_dir), "vendor", "spacetime_api_stubs")
if os.path.isdir(_stubs) and _stubs not in sys.path:
    sys.path.insert(0, _stubs)

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

# 3. verify the private-index package imported (else HAS_* are False downstream)
try:
    import aalyria.spacetime.api.common.auth  # noqa: F401
    print("spacetime-api: import OK")
except Exception as e:
    print("spacetime-api: NOT importable yet ->", type(e).__name__)
    print("  If pip installed it just above, do Runtime -> Restart session, then Run all.")
    print("  If pip errored above, check access to the private --extra-index-url registry.")

# 4. re-probe optional surfaces now that all roots (pip + vendored stubs) are on sys.path
import ngso_sls.spacetime as st
print("surface flags:", st.reprobe())   # HAS_NBI=True expected via vendored stubs

In [ ]:
# === Increment-0 capability probe ===
# Re-probes the installed spacetime-api (picks up an install done earlier in THIS runtime, so no
# kernel restart is needed) and reports which surfaces are available:
#   HAS_AUTH=True  -> spacetime-api is importable (auth) — required to connect at all.
#   HAS_MODEL=True -> NMTS entity pull + coverage path runs.
#   HAS_NBI=True   -> intents/routes pull runs.
import ngso_sls.spacetime as st

flags = st.reprobe()
print("Capability flags:", flags)

if not flags["HAS_AUTH"]:
    print("\nERROR: HAS_AUTH=False - spacetime-api is NOT importable in this runtime.")
    print("  1) Run the install cell above; check for pip errors from the private index.")
    print("  2) If it installed, do Runtime -> Restart session, then Run all.")
    print("  3) Still failing? Run the next (diagnostic) cell and share its output.")
elif not flags["HAS_MODEL"]:
    print(f"\nWARNING: HAS_MODEL=False - NMTS entity pull + coverage path will not run "
          f"(HAS_NBI={flags['HAS_NBI']} -> intents/provisioning still works).")
    print("  This build may not ship the Model service, or serves a different version.")
else:
    print("\nHAS_MODEL=True - full NMTS -> elements -> coverage path available.")

In [ ]:
# === (optional) Diagnostic: what did spacetime-api install as? ===
# Run this if HAS_AUTH / HAS_MODEL are False, to see whether the package installed and its layout.
import importlib, subprocess, sys
info = subprocess.run([sys.executable, "-m", "pip", "show", "spacetime-api"],
                      capture_output=True, text=True).stdout.strip()
print(info or "spacetime-api: NOT INSTALLED (pip show returned nothing)")
print("-" * 60)
for mod in ["aalyria", "aalyria.spacetime", "aalyria.spacetime.api",
            "aalyria.spacetime.api.common", "aalyria.spacetime.api.nbi.v1alpha",
            "aalyria.spacetime.api.model.v1", "aalyria.spacetime.api.model.v1alpha"]:
    try:
        importlib.import_module(mod)
        print("import OK  :", mod)
    except Exception as e:
        print("import FAIL:", mod, "->", type(e).__name__)
import ngso_sls.spacetime._deps as _d
print("-" * 60)
print("resolved MODEL_ROOT:", _d.MODEL_ROOT, "| NMTS_ROOT:", _d.NMTS_ROOT)
print("flags:", _d.reprobe())

In [ ]:
# === Connection form ===
# Values prefill from Colab Secrets (userdata); nothing is hardcoded here. Expected secret names:
#   SPACETIME_KEY_ID, SPACETIME_USER_ID, KEY_DATA_B64  (and optional SPACETIME_URL / SPACETIME_MODEL_URL)
# KEY_DATA_B64 = base64 of your private key file, produced locally with:
#   base64 < private_key.key | tr -d '\n'      (GNU: base64 -w0 private_key.key)
import os

def _secret(name, fallback=""):
    """Read from Colab userdata, then env, then fallback. Never raises, never prints the value."""
    try:
        from google.colab import userdata
        return userdata.get(name) or fallback
    except Exception:
        return os.environ.get(name, fallback)

_DEFAULT_URL = "https://fss01-demo.spacetime.aalyria.com:443"

try:
    import ipywidgets as widgets
    from IPython.display import display

    w_url  = widgets.Text(value=_secret("SPACETIME_URL", _DEFAULT_URL),
                          description="URL:", layout=widgets.Layout(width="620px"))
    w_key  = widgets.Password(value=_secret("SPACETIME_KEY_ID"), description="KEY_ID:",
                              layout=widgets.Layout(width="460px"))
    w_user = widgets.Password(value=_secret("SPACETIME_USER_ID"), description="USER_ID:",
                              layout=widgets.Layout(width="460px"))
    w_b64  = widgets.Password(value=_secret("KEY_DATA_B64"), description="KEY_DATA_B64:",
                              layout=widgets.Layout(width="620px"))
    w_murl = widgets.Text(value=_secret("SPACETIME_MODEL_URL"), description="MODEL_URL:",
                          placeholder="blank = use main URL", layout=widgets.Layout(width="620px"))
    w_mver = widgets.Dropdown(options=["v1", "v1alpha", "v0"], value="v1", description="model_ver:")

    display(widgets.VBox([
        widgets.HTML("<b>Connection</b> - prefilled from Colab Secrets. KEY_DATA_B64 is the "
                     "base64 of your key file (see the Setup section above). Never paste a raw key."),
        w_url, w_key, w_user, w_b64, w_murl, w_mver,
    ]))

    def _build_endpoint():
        return st.SpacetimeEndpoint(
            url=w_url.value.strip(),
            key_id=w_key.value.strip(),
            user_id=w_user.value.strip(),
            private_key_b64=(w_b64.value.strip() or None),
            model_url=(w_murl.value.strip() or None),
            model_version=w_mver.value,
        )

    print("Form ready (prefilled from Secrets where set). Run the smoke probe next.")

except ImportError:
    # Non-widget fallback (plain env / Colab userdata)
    def _build_endpoint():
        return st.SpacetimeEndpoint(
            url=_secret("SPACETIME_URL", _DEFAULT_URL),
            key_id=_secret("SPACETIME_KEY_ID"),
            user_id=_secret("SPACETIME_USER_ID"),
            private_key_b64=(_secret("KEY_DATA_B64") or None),
            model_url=(_secret("SPACETIME_MODEL_URL") or None),
            model_version=_secret("SPACETIME_MODEL_VERSION", "v1"),
        )

    print(f"ipywidgets not available; using env/userdata. URL={_secret('SPACETIME_URL', _DEFAULT_URL)}")

### Alternative: raw Store over `kubectl port-forward` (no robot key, no deploy)

If the runtime has cluster credentials (local Jupyter, or Colab with `gcloud`/`kubectl` set up),
the internal Store is reachable directly — engdoc pattern, same as `nbictl`/`storectl`:

```sh
kubectl --context=<ctx> port-forward svc/storage -n <namespace> 9999:9999
```

This reads intents AND the NMTS model with no key auth (the security boundary is kubectl
access). Skip this cell when using the key-authed endpoint above.


In [ ]:
# OPTIONAL — raw Store via port-forward (run the kubectl command above first).
# When enabled, this store is used by ALL following cells (pull, compare, record) —
# the key-authed GrpcEntityStore cell below is skipped automatically.
USE_RAW_STORE = False  #@param {type:"boolean"}
RAW_STORE_TARGET = "localhost:9999"  #@param {type:"string"}
store = None
if USE_RAW_STORE:
    from ngso_sls.spacetime.client import StorageEntityStore
    store = StorageEntityStore(RAW_STORE_TARGET)   # read-only: Get/GetEntities only
    print(f"raw Store @ {RAW_STORE_TARGET}: {len(store.list_intents())} intents")


In [ ]:
# === First-contact SMOKE PROBE (run this BEFORE the full pull) ===
# Staged diagnostic — capability flags -> auth/channel -> intents -> entities -> platform
# motion -> relationships, one RPC at a time with PASS/FAIL, so first contact is debuggable.
# It never logs your key. Also runnable as a CLI:
#   SPACETIME_URL=... SPACETIME_KEY_ID=... SPACETIME_USER_ID=... \
#   SPACETIME_PRIVATE_KEY_FILE=/path/key python -m ngso_sls.spacetime.probe
from ngso_sls.spacetime.probe import probe_endpoint, format_report

report = probe_endpoint(_build_endpoint())
print(format_report(report))

# How to read it / common fixes:
#   * capability flags all False      -> spacetime-api not installed; re-run the install cell.
#   * live store NOT built            -> auth/URL/key issue (KEY_ID, USER_ID, key-file path, :443).
#   * [FAIL] intents (Unimplemented)  -> instance exposes a different NBI (legacy NetOps) surface.
#   * [FAIL] entities (unknown service model.v1.Model) -> set model_ver = v1alpha (or v0) in the
#                                        form above and re-run this cell.
#   * platform motion "0 keplerian served" -> instance stores TLE/ephemeris motion; those are
#                                        skipped in Increment-1 (Sgp4 is the deferred follow-up).
# Proceed to the full pull (next cells) once at least intents + entities PASS.

In [ ]:
# === Build live store ===
# Requires HAS_AUTH + HAS_MODEL (spacetime-api installed and probe passed).
# The store object is read-only: no Create/Update/Delete is ever wired.

from ngso_sls.spacetime.client import GrpcEntityStore

if globals().get("store") is not None:
    print("Using the raw-Store connection from the cell above (USE_RAW_STORE=True).")
else:
    endpoint = _build_endpoint()
    print(f"Connecting to: {endpoint.url}  (model_version={endpoint.model_version})")
    store = GrpcEntityStore(endpoint)
    print("GrpcEntityStore ready — pulling...")

In [ ]:
# === Pull model + build elements + run coverage ===
from datetime import datetime, timezone
from ngso_sls.spacetime.pull import pull_and_cover
from ngso_sls.grids.aor import AORS

EPOCH_UTC  = datetime(2026, 1, 1, tzinfo=timezone.utc)
AOR        = AORS["India"]   # change to AORS["Global"] etc. as needed
CELL_RES   = 3               # H3 resolution (3=~100 km cells; 4=~30 km)
MIN_ELEV   = 25.0            # degrees
DURATION_S = 3600.0          # simulation window (s)
STEP_S     = 60.0            # time step (s)
K_VALUES   = (1, 2)          # k-coverage grades to compute

out = pull_and_cover(
    store, AOR,
    cell_res=CELL_RES,
    min_elev_user_deg=MIN_ELEV,
    duration_s=DURATION_S,
    step_s=STEP_S,
    epoch_utc=EPOCH_UTC,
    k_values=K_VALUES,
)

print(f"n_served  : {out['n_served']} Keplerian satellites")
print(f"skipped   : {len(out['skipped'])} platforms (TLE / external-system)")
for s in out["skipped"]:
    print(f"  skip  sat_id={s['sat_id']}  reason={s['reason']}")

print(f"\nPulled {out['n_served']} sats; ref_epoch={out['ref_epoch_s']:.0f} s")
cells = out["coverage"]["cells"]
avail_k1 = out["coverage"]["availability_by_k"][1]
print(f"Coverage cells: {len(cells)} | mean k=1 availability: {avail_k1.mean():.3f}")
print(f"Installed routes (hops): {len(out['routes'])}")

In [ ]:
# === Coverage viz ===
import matplotlib.pyplot as plt
from ngso_sls.viz.plots import plot_coverage_hexmap, plot_mbb_feasible_hexmap

print("k=1 coverage (availability):")
plot_coverage_hexmap(out["coverage"], title="Live Spacetime pull - k=1 coverage (India AOR)")
plt.show()

# MBB feasibility map (only present if continuity was requested in the pull)
if out["coverage"].get("mbb_feasible") is not None:
    print("MBB feasibility map:")
    plot_mbb_feasible_hexmap(out["coverage"], title="Live Spacetime pull - MBB feasibility (India AOR)")
    plt.show()

In [ ]:
# === Compare predicted access vs installed routes ===
# For each installed PathIntent hop, note whether its endpoints are served Keplerian
# satellites (not skipped), and write a validation report CSV.
import csv, pathlib

# Hop endpoints are NETWORK-NODE ids; served sats carry PLATFORM ids. Join via
# RK_CONTAINS(platform -> network-node), and classify ground endpoints (gateway/UT/PoP)
# as GROUND rather than "skipped".
RK_CONTAINS = 4
node2plat = {r.z: r.a for r in store.list_relationships()
             if int(getattr(r, "kind", -1)) == RK_CONTAINS
             and str(getattr(r, "z", "")).endswith("-network-node")}
served = {m["sat_id"] for m in out["meta"]}

def _cls(node_id):
    pid = node2plat.get(node_id, node_id)
    if pid in served:
        return "SERVED"
    return "GROUND" if not pid.startswith("satellite") else "SAT_UNSERVED"

routes = out["routes"]
report_rows = []
for hop in routes:
    src = hop["src"]
    dst = hop["dst"]
    cs, cd = _cls(src), _cls(dst)
    src_served, dst_served = cs == "SERVED", cd == "SERVED"
    status = ("SAT_UNSERVED" if "SAT_UNSERVED" in (cs, cd) else
              "BOTH_SERVED" if (src_served and dst_served) else
              "GROUND_HOP")
    report_rows.append({
        "src": src, "dst": dst, "src_if": hop.get("src_if"), "dst_if": hop.get("dst_if"),
        "src_served": src_served, "dst_served": dst_served, "status": status,
    })

report_path = pathlib.Path("validation_report.csv")
with open(report_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["src", "dst", "src_if", "dst_if",
                                           "src_served", "dst_served", "status"])
    writer.writeheader()
    writer.writerows(report_rows)

print(f"Validation report: {len(report_rows)} hops written to {report_path}")
for r in report_rows[:10]:
    print("  " + str(r["src"]) + " -> " + str(r["dst"]) + "  " + r["status"])
if len(report_rows) > 10:
    print(f"  ... and {len(report_rows) - 10} more")

In [ ]:
# === Record live snapshot for offline replay ===
# Serialises the store's read surface to a proto-free JSON projection.
# Replay offline with RecordedEntityStore(path).
# IMPORTANT: this file may contain sensitive topology - treat as confidential.
import ngso_sls.spacetime as st
import pathlib

snapshot_path = "spacetime_live_snapshot.json"
st.record(store, snapshot_path, intent_states=["INSTALLED"])
size_kb = pathlib.Path(snapshot_path).stat().st_size // 1024
print(f"Snapshot written: {snapshot_path}  ({size_kb} KB)")
print("Replay offline:")
print("  from ngso_sls.spacetime.recording import RecordedEntityStore")
print("  rec = RecordedEntityStore(" + repr(snapshot_path) + ")")